In [2]:


!pip install transformers datasets scikit-learn langgraph --quiet


# Imports

import os
import torch
import numpy as np
import pandas as pd
import logging
from datetime import datetime
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
from sklearn.metrics import f1_score
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver


# Device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# Logging

log_file = "classification_log.tsv"
open(log_file, "a").close()

logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.handlers.clear()

handler = logging.FileHandler(log_file)
handler.setFormatter(logging.Formatter("%(asctime)s\t%(message)s"))
logger.addHandler(handler)


# Config

NUM_LABELS = 28
TEMP_DATA_FILE = "for_update_data.tsv"
MODEL_NAME = "distilbert-base-uncased"

emotion_labels = [
    "admiration","amusement","anger","annoyance","approval","caring","confusion",
    "curiosity","desire","disappointment","disapproval","disgust","embarrassment",
    "excitement","fear","gratitude","grief","joy","love","nervousness","optimism",
    "pride","realization","relief","remorse","sadness","surprise","neutral"
]


# Load Dataset

train_df = pd.read_csv("/content/train.tsv", sep="\t", header=None)
dev_df   = pd.read_csv("/content/dev.tsv", sep="\t", header=None)
test_df  = pd.read_csv("/content/test.tsv", sep="\t", header=None)

def convert_to_multihot(df):
    def encode(label_str):
        multihot = np.zeros(NUM_LABELS, dtype=np.float32)
        if isinstance(label_str, str):
            for idx in map(int, label_str.split(",")):
                if idx < NUM_LABELS:
                    multihot[idx] = 1.0
        return multihot

    df = df.rename(columns={0: "text", 1: "label"})
    df["label"] = df["label"].apply(encode)
    return df

train_df = convert_to_multihot(train_df)
dev_df   = convert_to_multihot(dev_df)
test_df  = convert_to_multihot(test_df)

train_ds = Dataset.from_pandas(train_df)
dev_ds   = Dataset.from_pandas(dev_df)
test_ds  = Dataset.from_pandas(test_df)


# Tokenizer

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True)
dev_ds   = dev_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)


# Model

model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)
model.to(device)


# Metrics

def compute_metrics(pred):
    logits = pred.predictions
    labels = pred.label_ids
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)
    f1 = f1_score(labels.flatten(), preds.flatten(), average="micro")
    return {"f1_micro": f1}

# Training Arguments (FIXED)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    greater_is_better=True,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.evaluate(test_ds)


# Zero-Shot Fallback

zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1
)

# LangGraph Nodes

def tokenize_node(state):
    inputs = tokenizer(
        state["text"],
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    state["inputs"] = {k: v.tolist() for k, v in inputs.items()}
    return state

def predict_node(state):
    inputs = {k: torch.tensor(v).to(device) for k, v in state["inputs"].items()}
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(**inputs).logits).squeeze().cpu().numpy()
    state["probs"] = probs.tolist()
    return state

def fallback_node(state):
    probs = np.array(state["probs"])
    idx = probs.argmax()
    conf = probs[idx]
    state["confidence"] = float(conf)

    if conf < 0.5:
        result = zero_shot_classifier(state["text"], emotion_labels)
        state["emotion"] = result["labels"][0]
        state["source"] = "backup"
    else:
        state["emotion"] = emotion_labels[idx]
        state["source"] = "primary"
    return state

def save_node(state):
    if state["source"] == "backup":
        label_idx = emotion_labels.index(state["emotion"])
        row = pd.DataFrame([[state["text"], label_idx]])
        df = pd.read_csv(TEMP_DATA_FILE, sep="\t", header=None) if os.path.exists(TEMP_DATA_FILE) else pd.DataFrame()
        df = pd.concat([df, row], ignore_index=True)
        df.to_csv(TEMP_DATA_FILE, sep="\t", index=False, header=False)
    return state

def retrain_node(state):
    if not os.path.exists(TEMP_DATA_FILE):
        return state

    df = pd.read_csv(TEMP_DATA_FILE, sep="\t", header=None)
    if len(df) >= 10:
        main_df = pd.read_csv("/content/train.tsv", sep="\t", header=None)
        updated = pd.concat([main_df, df], ignore_index=True)
        updated.to_csv("/content/train.tsv", sep="\t", index=False, header=False)
        os.remove(TEMP_DATA_FILE)

        new_df = convert_to_multihot(updated)
        new_ds = Dataset.from_pandas(new_df).map(tokenize, batched=True)
        trainer.train_dataset = new_ds
        trainer.train()

    return state

def output_node(state):
    logging.info(
        f"{state['text']}\t{state['emotion']}\t{state['confidence']:.2f}\t{state['source']}"
    )
    return state

# Build LangGraph

graph = StateGraph(dict)
graph.add_node("Tokenize", tokenize_node)
graph.add_node("Predict", predict_node)
graph.add_node("Fallback", fallback_node)
graph.add_node("Save", save_node)
graph.add_node("Retrain", retrain_node)
graph.add_node("Output", output_node)

graph.set_entry_point("Tokenize")
graph.add_edge("Tokenize", "Predict")
graph.add_edge("Predict", "Fallback")
graph.add_edge("Fallback", "Save")
graph.add_edge("Save", "Retrain")
graph.add_edge("Retrain", "Output")
graph.add_edge("Output", END)

app = graph.compile()


# Interactive Loop

while True:
    text = input("\nEnter a sentence (or 'exit'): ").strip()
    if text.lower() == "exit":
        print("Exiting...")
        break

    result = app.invoke({"text": text})
    print(f"Emotion: {result['emotion']} | Source: {result['source']}")


Using device: cuda


/usr/local/lib/python3.12/dist-packages/datasets/table.py:719: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  return cls(pa.Table.from_pandas(*args, **kwargs))


Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0



Enter a sentence (or 'exit'): I DONT KNOW WHAT TO DO?
Emotion: confusion | Source: primary

Enter a sentence (or 'exit'): WHY DID YOU DO THIS?
Emotion: curiosity | Source: primary

Enter a sentence (or 'exit'): ARE YOU OK?
Emotion: caring | Source: backup

Enter a sentence (or 'exit'): THIS MOVIE IS OSM.
Emotion: neutral | Source: primary

Enter a sentence (or 'exit'): exit
Exiting...
